# Nueva sección

In [4]:
import pandas as pd

# 1. Cargar los archivos CSV por separado
df_2019 = pd.read_csv("Divvy_Trips_2019_Q1.csv")
df_2020 = pd.read_csv("Divvy_Trips_2020_Q1.csv")

# 2. Cambiar los nombres de las columnas de 2019 para que coincidan con las de 2020
df_2019 = df_2019.rename(
    columns={
        "trip_id": "ride_id",
        "start_time": "started_at",
        "end_time": "ended_at",
        "from_station_name": "start_station_name",
        "from_station_id": "start_station_id",
        "to_station_name": "end_station_name",
        "to_station_id": "end_station_id",
        "usertype": "member_casual", })

# 3. Cambiar los nombres de los usuarios en 2019 (Subscriber->member, Customer->casual)
df_2019["member_casual"] = df_2019["member_casual"].replace(
    {"Subscriber": "member", "Customer": "casual"}
)
# 4. Seleccionar las columnas que están en ambos datasets
columnas_comunes = [
    "ride_id",
    "started_at",
    "ended_at",
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id",
    "member_casual",
    "ride_length",
    "weekday"
]

df_2019_sub = df_2019[columnas_comunes]
df_2020_sub = df_2020[columnas_comunes]

# 5. Juntar ambos datasets en uno solo (all_trips)
all_trips = pd.concat([df_2019_sub, df_2020_sub], ignore_index=True)

# 1. Convertir las fechas usando format='mixed' para que Pandas maneje los cambios de formato
all_trips["started_at"] = pd.to_datetime(
    all_trips["started_at"], format="mixed", dayfirst=True
)
all_trips["ended_at"] = pd.to_datetime(
    all_trips["ended_at"], format="mixed", dayfirst=True
)

# 2. Calcular la duración en minutos
all_trips["ride_length"] = (
    all_trips["ended_at"] - all_trips["started_at"]
).dt.total_seconds() / 60

# 3. Crear el día de la semana
all_trips["weekday"] = all_trips["started_at"].dt.dayofweek

print("¡Listo! Fechas convertidas sin errores.")
display(
    all_trips[["started_at", "ended_at", "ride_length", "weekday"]].head()
)
# 1. Filtrar para quedarnos solo con viajes mayores a 0 minutos y menores a 1 día (1440 minutos)
all_trips_clean = all_trips[
    (all_trips["ride_length"] > 0) & (all_trips["ride_length"] <= 1440)
].copy()

# 2. Comprobar cuántos datos teníamos antes y cuántos nos quedan limpios
print(f"Filas originales: {len(all_trips)}")
print(f"Filas limpias: {len(all_trips_clean)}")

# 3. Volver a correr el describe solo para ride_length para ver cómo mejoró
print("panorama general")
display(all_trips_clean[["ride_length"]].describe())

print("recuento")
display(all_trips_clean['member_casual'].value_counts())

# Resumen general por tipo de usuario (cantidad de viajes, promedio y mediana de duración)
resumen_usuario = all_trips_clean.groupby('member_casual')['ride_length'].agg(
    total_viajes='count',
    promedio_min='mean',
    mediana_min='median'
)
display(resumen_usuario)

# Resumen por tipo de usuario y día de la semana (promedio y cantidad de viajes)
resumen_dias = (
    all_trips_clean.groupby(["member_casual", "weekday"])["ride_length"]
    .agg(promedio_min="mean", total_viajes="count")
    .reset_index()
)
display(resumen_dias)

# Exportar ambas tablas a CSV
resumen_usuario.reset_index().to_csv("resumen_usuario.csv", index=False)
resumen_dias.to_csv("resumen_dias.csv", index=False)

# Descargar los archivos a tu computador desde Colab
from google.colab import files

files.download("resumen_usuario.csv")
files.download("resumen_dias.csv")


¡Listo! Fechas convertidas sin errores.


,started_at,ended_at,ride_length,weekday
0,2019-01-01 00:04:00,2019-01-01 00:11:00,7.0,1
1,2019-01-01 00:08:00,2019-01-01 00:15:00,7.0,1
2,2019-01-01 00:13:00,2019-01-01 00:27:00,14.0,1
3,2019-01-01 00:13:00,2019-01-01 00:43:00,30.0,1
4,2019-01-01 00:14:00,2019-01-01 00:20:00,6.0,1


Filas originales: 791956
Filas limpias: 785390
panorama general


,ride_length
count,785390.000000
mean,13.763767
std,31.150944
min,1.000000
25%,6.000000
50%,9.000000
75%,15.000000
max,1435.000000


recuento


,count
member_casual,
member,717727
casual,67663


... (2 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>